# Bayesian die‑swell inference — a mathematical walkthrough

This notebook re‑runs your **Oldroyd‑B + pressure** script line by line and explains,
with the math, *exactly* what each call computes. The equations below reflect the
actual implementation in the `packages/` module (ROM = POD+RBF surrogate, pressure =
GPR, `emcee` sampler), not a generic template.

**The problem in one sentence.** From a single noisy, measured free‑surface (die‑swell)
curve — reduced to its peak height — plus a few noisy pressure readings, infer the
Oldroyd‑B material parameters $\theta=(\lambda,\beta,\eta_0)$ and the noise level
$\sigma_n$, in the Bayesian sense: produce the full posterior $P(\theta,\sigma_n\mid
\mathcal D)$.

**Pipeline.**

1. `ROMCurve4BayesianInference(...)` — declare the model: unknowns, priors, forward surrogate.
2. `load_data(...)` — build the observation $\mathcal D$ (thin, add noise, calibrate the $\sigma$‑prior, load pressure).
3. `build_rom()` — train the cheap forward map $\theta \mapsto (\text{curve},\ \text{pressure})$.
4. `run_mcmc(...)` — sample the posterior with `emcee`.
5. `compute_N1()` — push the posterior mean through the Oldroyd‑B constitutive law to get $N_1(\dot\gamma)$.
6. `plot_posterior_predictive()` / `plot_corner()` — check the fit and view the posterior.

---
## 0. Notation

| symbol | meaning | in code |
|---|---|---|
| $\theta=(\lambda,\beta,\eta_0)$ | relaxation time, solvent fraction, zero‑shear viscosity | material params |
| $\sigma_n$ | measurement noise std (shared by curve & pressure) | `sigma_noise` |
| $y(x)$ | free‑surface height vs. axial coordinate $x$ | `curve4_y` |
| $h=\max_x y(x)$ | **swell height** (the scalar observable here) | `_swell_height` |
| $p\in\mathbb R^{5}$ | pressure vector (5 gauge points) | `pressure` |
| $\hat y(\theta),\ \hat p(\theta)$ | surrogate predictions | ROM / pressure GPR |
| $\mathcal D=\{h^{\mathrm{obs}},\,p^{\mathrm{obs}}\}$ | the data | observations |

The unknown vector sampled by MCMC is $\phi=(\lambda,\beta,\eta_0,\sigma_n)\in\mathbb R^{4}$.

## 1. Imports and path setup

Nothing mathematical here — this just puts the repository root on `sys.path` so
`from packages import ...` works. Run this notebook from the `examples/` folder so that
`..` resolves to the project root.

In [ ]:
import sys
import os
from pathlib import Path

CURRENT_DIR = os.getcwd()
FAXEN_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, '..'))
sys.path.append(FAXEN_ROOT)
from packages import ROMCurve4BayesianInference

## 2. Declaring the model

This constructor doesn't compute anything yet; it **fixes the probabilistic model**.

**Unknowns.** `model="oldroyd"` with `use_pressure=True, augment_eta0=True` promotes
$\eta_0$ to a third inferred material parameter, so

$$\theta=(\lambda,\beta,\eta_0),\qquad \phi=(\lambda,\beta,\eta_0,\sigma_n),\qquad \dim\phi = 4 .$$

(`sigma_bias=None` means there is **no** model‑discrepancy term — the only stochastic
part of the likelihood is i.i.d. Gaussian noise with std $\sigma_n$.)

**Priors.** Material parameters get **uniform** priors on the boxes you passed, and the
noise gets an **exponential** prior (rate set later from the data):

$$
\lambda\sim\mathcal U(0.5,\,9.0),\quad
\beta\sim\mathcal U(0.1,\,0.98),\quad
\eta_0\sim\mathcal U(0.5,\,2.0),\quad
\sigma_n\sim\mathrm{Exp}(b).
$$

So the (log) prior evaluated in `priors.py` is

$$
\log p(\theta,\sigma_n)=
-\!\!\sum_{k\in\{\lambda,\beta,\eta_0\}}\!\!\log(h_k-l_k)\;+\;\log b - b\,\sigma_n,
$$

returning $-\infty$ whenever any parameter leaves its box or $\sigma_n\le 0$.

**Observable.** `mode="swell_height"` means the curve is compared **only through its peak
height** $h=\max_x y(x)$, not point‑by‑point.

**Ground truth** `true_theta=[1.0, 0.1, 0.2]` $=(\lambda,\beta,\eta_0)$ is stored only to
draw reference lines; it is not used by the inference.

In [ ]:
model = ROMCurve4BayesianInference(
    swell_root=Path(FAXEN_ROOT),
    model="oldroyd",  # "oldroyd", " ", or "ptt"
    train_data_rels=["datas/oldroyd-rom3/curve4_y_all.txt"],
    mode="swell_height",  # or "full_curve"
    rom_method="rbf",
    scaler="minmax",
    eps=1e-6,
    sigma_noise_percent=0.5,
    lambda_bounds = (0.5, 9.0),
    beta_bounds = (0.1, 0.98),
    eta0_bounds = (0.5, 2.0),
    thin=2,
    sigma_bias=None,
    use_pressure = True,
    augment_eta0=True,
    pressure_filename= "pressure_all.txt",
    true_theta=[1.0, 0.1, 0.2])

## 3. Building the observation — `load_data`

This constructs the data $\mathcal D$ from one high‑fidelity (FOM) curve and its pressure
file, and calibrates the noise prior. Four things happen (`data_io.py`):

**(a) Thinning.** Keep every `thin=2`‑th point: index set $I=\{0,2,4,\dots\}$, giving the
clean curve $y^{\text{clean}}=y_{\text{FOM}}[I]$ on grid $x=x_{\text{FOM}}[I]$.

**(b) Noise‑prior calibration.** With the maximum swelling displacement

$$d_{\max}=\max_i y^{\text{clean}}_i-1,$$

the exponential‑prior rate is set so its mean noise is $10\%$ of that displacement:

$$b=\frac{1}{0.10\,d_{\max}},\qquad \mathbb E[\sigma_n]=1/b = 0.10\,d_{\max}.$$

**(c) Synthetic measurement noise.** The *observed* curve adds Gaussian noise at
`sigma_noise_percent=0.5`\% of $d_{\max}$:

$$\sigma_\star=\tfrac{0.5}{100}\,d_{\max},\qquad
y^{\text{obs}}=y^{\text{clean}}+\varepsilon,\quad \varepsilon\sim\mathcal N(0,\sigma_\star^2 I).$$

The scalar observable is then $h^{\text{obs}}=\max_i y^{\text{obs}}_i$.

**(d) Pressure observation.** The pressure vector is loaded and given the *same* noise std:

$$p^{\text{obs}}=p^{\text{clean}}+\varepsilon_p,\qquad \varepsilon_p\sim\mathcal N(0,\sigma_\star^2 I),\quad p^{\text{obs}}\in\mathbb R^{5}.$$

Here $U_{\text{avg}}=0.1$ is recorded for later (it sets the wall shear rate in `compute_N1`).

In [ ]:
model.load_data(filename="datas/oldroyd-rom3/giesekus0/curve4_y.txt",
                u_avg_obs=0.1, pressure_filename="datas/oldroyd-rom3/giesekus0/pressure.txt")

### Look at the data

`plot_data()` just draws the clean curve (line) and the noisy observed points
$y^{\text{obs}}$ vs. $x$. No computation of interest — it's a sanity check on
steps (a)–(c) above.

In [ ]:
model.plot_data()

## 4. Training the forward surrogate — `build_rom`

The likelihood must evaluate the forward map $\theta\mapsto(\text{curve},\text{pressure})$
thousands of times, so the expensive PDE solver is replaced by two cheap surrogates.

### 4.1 Curve ROM = POD (SVD) basis + RBF interpolation

Let the training snapshots be $Y\in\mathbb R^{N\times M}$ ($N$ curves, $M$ points), with
parameters $\{p_i\}=\{(\lambda_i,\beta_i,\eta_{0,i})\}$.

**Proper Orthogonal Decomposition.** Subtract the mean $\bar y=\frac1N\sum_i y_i$, center
$\tilde Y = Y-\mathbf 1\bar y^\top$, and take the SVD

$$\tilde Y^\top = U\Sigma V^\top .$$

Keep the leading $r$ modes by a relative‑energy cutoff $\varepsilon=10^{-6}$,

$$r=\#\{k:\ \sigma_k/\sigma_1\ge\varepsilon\},\qquad \Phi=U_{:,1:r}\in\mathbb R^{M\times r},$$

and project each snapshot onto the basis to get POD coefficients $A=\tilde Y\,\Phi\in\mathbb R^{N\times r}$.

**Input scaling (`minmax`).** $\;x=\dfrac{p-p_{\min}}{p_{\max}-p_{\min}}\in[0,1]^3$.

**RBF interpolation of the coefficients (`rom_method="rbf"`, quintic).** Fit weights $w_j$ so that
a radial‑basis interpolant reproduces every training coefficient vector,

$$\hat a(x)=\sum_{j=1}^{N} w_j\,\varphi(\lVert x-x_j\rVert)+(\text{low‑order poly}),\qquad \varphi(r)=r^5,$$

with $\hat a(x_i)=A_i$. **Prediction** for a new $\theta$ then costs one interpolation +
one matrix product:

$$\boxed{\;\hat y(\theta)=\bar y+\Phi\,\hat a\big(\mathrm{scale}(\theta)\big)\;}$$

### 4.2 Pressure surrogate = Gaussian Process regression

`use_pressure=True` also trains a **direct GPR** $\hat p(\theta)$ from the scaled inputs to
the 5‑point pressure vector (no POD — pressure is already low‑dimensional). With an
anisotropic squared‑exponential kernel

$$k(x,x')=c\,\exp\!\Big(-\tfrac12\textstyle\sum_{d}(x_d-x'_d)^2/\ell_d^2\Big),$$

the GP posterior mean used as the predictor is

$$\hat p(\theta)=K_{*}\,(K+\alpha I)^{-1}\,P,\qquad \alpha=10^{-10},$$

where $K$ is the train–train kernel matrix, $K_*$ the test–train row, and $P$ the training
pressures. Because `augment_eta0=True`, $\eta_0$ enters *both* surrogates; in practice it is
only weakly identifiable from the curve and is **pinned mainly by this pressure channel**.

In [ ]:
model.build_rom()

## 5. The prior — `plot_prior`

Draws the prior densities you declared in step 2:

- flat densities $1/(h_k-l_k)$ on $[l_k,h_k]$ for $\lambda,\beta,\eta_0$;
- the exponential density $b\,e^{-b\sigma_n}$ for the noise, with the data‑calibrated
  rate $b=1/(0.10\,d_{\max})$ from step 3(b).

These are exactly the factors that enter $\log p(\theta,\sigma_n)$.

In [ ]:
model.plot_prior()

## 6. Posterior sampling — `run_mcmc`

Now Bayes' theorem is applied. The target (up to an additive constant) is

$$
\log P(\phi\mid\mathcal D)=\underbrace{\log p(\theta,\sigma_n)}_{\text{prior, step 2/5}}
\;+\;\underbrace{\log\mathcal L_{\text{curve}}(\phi)+\log\mathcal L_{\text{press}}(\phi)}_{\text{likelihood}} .
$$

### 6.1 The likelihood

Because `mode="swell_height"`, the curve residual is the **scalar** peak‑height mismatch

$$r(\theta)=h^{\text{obs}}-\max_x\hat y(\theta),$$

and with `sigma_bias=None` it is i.i.d. Gaussian, so (`likelihood.py`, $n=1$):

$$
\log\mathcal L_{\text{curve}}
=-\tfrac12\!\left[\frac{r(\theta)^2}{\sigma_n^{2}}+\log\!\big(2\pi\sigma_n^{2}\big)\right].
$$

The pressure channel adds a Gaussian term over its $m=5$ points, sharing the **same** $\sigma_n$:

$$
r_p(\theta)=p^{\text{obs}}-\hat p(\theta)\in\mathbb R^{5},\qquad
\log\mathcal L_{\text{press}}
=-\tfrac12\!\left[\frac{\lVert r_p(\theta)\rVert^2}{\sigma_n^{2}}+m\log(2\pi)+2m\log\sigma_n\right].
$$

Total: $\ \log\mathcal L=\log\mathcal L_{\text{curve}}+\log\mathcal L_{\text{press}}$. (Any
non‑finite evaluation returns $-\infty$ so the sampler simply rejects that proposal.)

### 6.2 The sampler (`emcee`, affine‑invariant ensemble)

- **Initialization:** `nwalkers=10` walkers are drawn from the prior, giving $\phi_0\in\mathbb R^{10\times 4}$.
- **Propagation:** the affine‑invariant *stretch move* proposes, for walker $\phi_i$ using a random partner $\phi_j$,

$$\phi_i' = \phi_j + z\,(\phi_i-\phi_j),\qquad z\sim g(z)\propto \tfrac{1}{\sqrt z}\ \text{on}\ [1/a,a],$$

accepted with probability $\min\!\big(1,\ z^{\,d-1}\,e^{\log P(\phi_i')-\log P(\phi_i)}\big)$, $d=4$.
- **Run length:** `nsteps=10000`; the first $30\%$ ($3000$ steps) are discarded as burn‑in.
- **Outputs stored:** the flattened post‑burn samples $\{\phi^{(k)}\}$ (`model.samples`), the
  MAP estimate $\hat\phi_{\text{MAP}}=\arg\max_k \log P(\phi^{(k)})$ (`model.map_theta`), and
  per‑parameter mean/std/95\% CI plus the **Gelman–Rubin** convergence diagnostic

$$\hat R=\sqrt{\frac{\frac{n-1}{n}W+\frac1n B}{W}}\ \xrightarrow{\text{converged}}\ 1,$$

with within‑ and between‑chain variances $W,B$. (`run_mcmc` seeds NumPy's global RNG from
`self.seed`, so this whole run is reproducible.)

In [ ]:
model.run_mcmc(nwalkers=10, nsteps=10000)

## 7. Rheology postprocessing — `compute_N1`

This takes the **posterior mean** material parameters
$\bar\theta=(\bar\lambda,\bar\beta,\bar\eta_0)=\mathbb E[\theta\mid\mathcal D]$ and evaluates
the first normal‑stress difference $N_1=\tau_{xx}-\tau_{yy}$ of the Oldroyd‑B fluid.

**Shear rate.** By default it uses the axisymmetric wall shear rate at $U_{\text{avg}}=0.1$,
radius $R=1$:

$$\dot\gamma_w=\frac{4\,U_{\text{avg}}}{R}=0.4 .$$

**Oldroyd‑B decomposition.** From $(\bar\lambda,\bar\beta,\bar\eta_0)$,

$$\eta_s=\bar\beta\,\bar\eta_0\ \ (\text{solvent}),\qquad
\eta_p=(1-\bar\beta)\,\bar\eta_0\ \ (\text{polymer}),\qquad
G=\eta_p/\bar\lambda\ \ (\text{modulus}).$$

**Constitutive model** (`rheology.py`). The polymer conformation tensor $\mathbf C$ obeys
the upper‑convected Maxwell evolution, integrated to steady state:

$$\dot{\mathbf C}=\mathbf L\,\mathbf C+\mathbf C\,\mathbf L^\top-\frac{1}{\bar\lambda}\big(\mathbf C-\mathbf I\big),
\qquad
\boldsymbol\tau=\underbrace{G(\mathbf C-\mathbf I)}_{\text{polymer}}+\underbrace{\eta_s(\mathbf L+\mathbf L^\top)}_{\text{solvent}},$$

with the shear velocity gradient $\mathbf L$ having the single entry $L_{xy}=\dot\gamma_w$.
The code solves the steady state numerically (Newton), then reports
$N_1=\tau_{xx}-\tau_{yy}$. For steady shear this reproduces the classical closed form

$$\boxed{\,N_1 = 2\,\eta_p\,\bar\lambda\,\dot\gamma_w^{2}\,},\qquad
\tau_{xy}=\bar\eta_0\,\dot\gamma_w,\qquad
\Psi_1=\frac{N_1}{\dot\gamma_w^{2}}=2\,\eta_p\,\bar\lambda .$$

So $N_1$ is the model's prediction of elastic normal stress at the inferred parameters —
the physically meaningful quantity the whole inference is ultimately after.

In [ ]:
model.compute_N1()

## 8. Posterior predictive check — `plot_posterior_predictive`

Does the inferred posterior actually reproduce the data (with its uncertainty)? For a
subset of posterior draws $\{(\theta^{(k)},\sigma_n^{(k)})\}$ the code forms **replicated**
datasets and summarizes their spread (`plotting.py::_predictive_summary`).

For each draw it evaluates the surrogate curve $g^{(k)}=\hat y(\theta^{(k)})$ and, since there
is no discrepancy term, adds pure measurement noise:

$$y^{(k)}_{\text{rep}} = g^{(k)} + \sigma_n^{(k)}\,z^{(k)},\qquad z^{(k)}\sim\mathcal N(0,I).$$

The shaded band is the pointwise $[\,(1-c)/2,\ (1+c)/2\,]$ percentile envelope of
$\{y^{(k)}_{\text{rep}}\}$ with $c=2\Phi(1.96)-1\approx95\%$, and the line is the mean
$\overline{g}=\frac1K\sum_k g^{(k)}$. It also prints calibration diagnostics — the
empirical **coverage** (fraction of observed points inside the band, ideally $\approx c$)
and standardized residuals $z_i=(y^{\text{obs}}_i-\overline g_i)/\mathrm{std}_k(y^{(k)}_{\text{rep},i})$
summarized as $\mathrm{rms}\,z$, $\max|z|$, mean $z$.

Because `use_pressure=True`, it then draws the **pressure** posterior‑predictive band the same
way, using $\hat p(\theta^{(k)})+\sigma_n^{(k)} z$.

> Note: inference used only the *swell height*, but this diagnostic plots the full‑curve
> band — a stringent visual check that the height‑only fit still explains the whole curve.

In [ ]:
model.plot_posterior_predictive()

## 9. Posterior geometry — `plot_corner`

The corner plot shows the joint posterior $P(\theta\mid\mathcal D)$: the diagonals are the
1‑D marginals $P(\lambda\mid\mathcal D),P(\beta\mid\mathcal D),P(\eta_0\mid\mathcal D)$
(with the flat prior drawn in red for reference and the $2.5\%/97.5\%$ credible bounds as
dotted lines), and the off‑diagonals are the pairwise densities — their tilt/shape reveals
**parameter correlations and identifiability** (e.g. a $\lambda$–$\eta_0$ ridge means the
data constrain a combination better than either alone).

`true_param=False` suppresses the ground‑truth marker (appropriate here, since
`giesekus0/…` is a *cross‑model* test curve, not an Oldroyd‑B curve at `true_theta`).

In [ ]:
model.plot_corner(true_param = False)

## Summary — what the script computed

$$
\underbrace{\text{FOM curve + pressure}}_{\text{load\_data}}
\;\xrightarrow{\ \text{build\_rom}\ }\;
\underbrace{\hat y(\theta),\ \hat p(\theta)}_{\text{cheap surrogates}}
\;\xrightarrow{\ \text{run\_mcmc}\ }\;
\underbrace{P(\lambda,\beta,\eta_0,\sigma_n\mid\mathcal D)}_{\text{posterior}}
\;\xrightarrow{\ \text{compute\_N1}\ }\;
N_1(\dot\gamma_w).
$$

1. **Model** — unknowns $\phi=(\lambda,\beta,\eta_0,\sigma_n)$; uniform priors on the material
   box, exponential prior on $\sigma_n$.
2. **Data** — one thinned, $0.5\%$‑noised swell curve reduced to its peak $h^{\text{obs}}$,
   plus a 5‑point noised pressure vector $p^{\text{obs}}$.
3. **Forward map** — POD+RBF for the curve, GPR for the pressure.
4. **Likelihood** — Gaussian in the height residual and the pressure residual, sharing $\sigma_n$.
5. **Posterior** — 10‑walker × 10000‑step `emcee`, 30\% burn‑in, with $\hat R$ convergence.
6. **Physics readout** — $N_1=2\eta_p\bar\lambda\dot\gamma_w^2$ at the posterior mean, plus
   predictive and corner diagnostics.